In [ ]:
# 1. Setup
import os
import pandas as pd
import numpy as np

DATA_RAW_PATH = "../data/raw/"
DATA_PROCESSED_PATH = "../data/processed/"

os.makedirs(DATA_PROCESSED_PATH, exist_ok=True)


In [7]:
# 2. Load raw data
# Example: adjust filename to your downloaded Kaggle CSV
raw_file = os.path.join(DATA_RAW_PATH, "SAML-D_reduced.csv")
df_raw = pd.read_csv(raw_file)
df_raw.head()


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
1,10:35:31,2022-10-07,5119661534,9734073275,2342.31,UK pounds,UK pounds,UK,UK,Debit card,0,Normal_Small_Fan_Out
2,10:35:46,2022-10-07,3709430533,9172843471,5274.76,UK pounds,UK pounds,UK,UK,Debit card,0,Normal_Fan_Out
3,10:36:05,2022-10-07,1203252958,8500212178,2438.30,UK pounds,Mexican Peso,UK,Mexico,Cross-border,0,Normal_Group
4,10:36:34,2022-10-07,7669236826,6044424887,8560.28,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_Out


In [8]:
# 3. Inspect schema and basic info
df_raw.info()
df_raw.describe(include="all").transpose()
df_raw["Is_laundering"].value_counts(normalize=True)


<class 'pandas.DataFrame'>
RangeIndex: 950486 entries, 0 to 950485
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Time                    950486 non-null  str    
 1   Date                    950486 non-null  str    
 2   Sender_account          950486 non-null  int64  
 3   Receiver_account        950486 non-null  int64  
 4   Amount                  950486 non-null  float64
 5   Payment_currency        950486 non-null  str    
 6   Received_currency       950486 non-null  str    
 7   Sender_bank_location    950486 non-null  str    
 8   Receiver_bank_location  950486 non-null  str    
 9   Payment_type            950486 non-null  str    
 10  Is_laundering           950486 non-null  int64  
 11  Laundering_type         950486 non-null  str    
dtypes: float64(1), int64(3), str(8)
memory usage: 87.0 MB


Is_laundering
0    0.998936
1    0.001064
Name: proportion, dtype: float64

# 4. Data cleaning
# - handle missing values
# - standardise column names
# - convert dtypes (datetime, numeric)
df = df_raw.copy()

# Example conversions – adjust to exact column names
df["Datetime"] = pd.to_datetime(df["Date"])
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

# Simple missing value handling (tune as needed)
df = df.dropna(subset=["amount", "sender_id", "receiver_id"])

df.info()


In [9]:
df = df_raw.copy()

In [17]:
df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"])
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

In [14]:
df = df.dropna(subset=["Amount", "Sender_account", "Receiver_account"])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 950486 entries, 0 to 950485
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   Time                    950486 non-null  str           
 1   Date                    950486 non-null  str           
 2   Sender_account          950486 non-null  int64         
 3   Receiver_account        950486 non-null  int64         
 4   Amount                  950486 non-null  float64       
 5   Payment_currency        950486 non-null  str           
 6   Received_currency       950486 non-null  str           
 7   Sender_bank_location    950486 non-null  str           
 8   Receiver_bank_location  950486 non-null  str           
 9   Payment_type            950486 non-null  str           
 10  Is_laundering           950486 non-null  int64         
 11  Laundering_type         950486 non-null  str           
 12  Datetime                950486 non-null  

In [15]:
# 5. Basic integrity checks
assert df["Amount"].ge(0).all(), "Found negative amounts"
assert df["Is_laundering"].isin([0, 1]).all(), "Invalid label values"


In [16]:
# 6. Save cleaned transaction-level dataset
processed_file = os.path.join(DATA_PROCESSED_PATH, "SAML-D_clean.csv")
df.to_csv(processed_file, index=False)
processed_file


'./data/processed/SAML-D_clean.csv'